# Roofline Cost Model, Hardware Counters & Calibration

This tutorial covers three features that improve the accuracy and observability
of GPU cost estimates in trovolut:

1. **Roofline-aware cost model** -- `emulated_time()` now uses
   `max(compute_time, memory_time)` instead of assuming purely compute-bound workloads
2. **NVML hardware counters** -- real-time VRAM, power, utilization, and temperature
   via `pynvml` (graceful no-op without a GPU)
3. **Calibration infrastructure** -- fit linear correction factors between emulator
   predictions and actual hardware measurements

All examples run on CPU using the `gpu-mp-emulator` backend -- no GPU required.

In [ ]:
import numpy as np
import trovolut as trv

trv.set_backend('gpu-mp-emulator')

## 1. Roofline-Aware Cost Model

### Background

The classical roofline model states that execution time is bounded by
**two ceilings**: the compute throughput and the memory bandwidth.
For a workload with $F$ FLOPs and $B$ bytes of memory traffic:

$$
T = \max\!\left(\frac{F}{\text{throughput}},\;\frac{B}{\text{bandwidth}}\right)
$$

The **arithmetic intensity** $I = F / B$ (FLOPs per byte) determines which
ceiling dominates. Below the ridge point, the workload is memory-bound;
above it, compute-bound.

Previously, `emulated_time()` only computed $F / \text{throughput}$. Now it
also tracks `bytes_accessed` and applies the roofline formula.

### 1.1 Hardware Setup

In [ ]:
from trovolut.backends.gpu_mp_emulator.hardware import gpus

gpu = gpus.A100
print(f"GPU: {gpu.name}")
print(f"  FP32 throughput:    {gpu.fp32_flops / 1e12:.1f} TFLOPS")
print(f"  FP16 throughput:    {gpu.fp16_flops / 1e12:.0f} TFLOPS")
print(f"  Memory bandwidth:   {gpu.memory_bandwidth / 1e12:.1f} TB/s")
print(f"  Ridge point (FP32): {gpu.fp32_flops / gpu.memory_bandwidth:.1f} FLOP/B")

### 1.2 Bytes Accessed in Cost Models

Each decomposition method now reports `bytes_accessed` -- the total bytes
transferred to/from memory during the computation.

In [ ]:
from trovolut.backends.gpu_mp_emulator.measure import DecompositionCostModel

n = 256  # matrix dimension

for name, fn, kwargs in [
    ("Exact (expm)", DecompositionCostModel.exact_expm_cost, {"n": n, "precision_bits": 32}),
    ("Taylor (order=20)", DecompositionCostModel.taylor_cost, {"n": n, "order": 20, "precision_bits": 32}),
    ("Krylov (dim=30)", DecompositionCostModel.krylov_cost, {"n": n, "krylov_dim": 30, "precision_bits": 32}),
    ("Sparse (nnz=1000)", DecompositionCostModel.sparse_cost, {"n": n, "nnz": 1000, "order": 20, "precision_bits": 32}),
]:
    r = fn(**kwargs)
    ai = r['flops'] / r['bytes_accessed'] if r['bytes_accessed'] > 0 else float('inf')
    print(f"{name:25s}  FLOPs={r['flops']:>12,}  bytes={r['bytes_accessed']:>12,}  AI={ai:.2f} FLOP/B")

### 1.3 Roofline in Action

Let's compare the old (compute-only) and new (roofline) execution time estimates
for a Trotter evolution at different precisions.

In [ ]:
from trovolut.backends.gpu_mp_emulator.dynamics import MPTrotterEvolution
from trovolut.backends.gpu_mp_emulator.utils import Precision

# Build a small Hamiltonian (2 terms, 64x64)
n = 64
rng = np.random.default_rng(42)
A = rng.standard_normal((n, n)) + 1j * rng.standard_normal((n, n))
H = [(A + A.conj().T) / 2, (A.T + A.conj()) / 4]

print(f"{'Precision':<12} {'Compute (s)':>12} {'Memory (s)':>12} {'Roofline (s)':>13} {'Bottleneck':>12}")
print("-" * 65)

for prec in [Precision.FP64, Precision.FP32, Precision.FP16]:
    evo = MPTrotterEvolution(trotter_steps=100, precision_schedule=prec)
    cost = evo.cost_profile(H)

    # Compute-only time (old behavior)
    compute_s = sum(
        flops / gpu.spec({
            'float64': 'fp64_flops', 'float32': 'fp32_flops',
            'float16': 'fp16_flops', 'bfloat16': 'bf16_flops',
        }.get(pk, 'fp32_flops'))
        for pk, flops in cost.flops_by_precision.items()
    )
    # Memory time
    memory_s = cost.total_bytes_accessed / gpu.memory_bandwidth if cost.total_bytes_accessed > 0 else 0
    # Roofline time (new behavior)
    roofline_s = cost.emulated_time(gpu)
    bottleneck = "memory" if memory_s > compute_s else "compute"

    print(f"{prec.value:<12} {compute_s:>12.2e} {memory_s:>12.2e} {roofline_s:>13.2e} {bottleneck:>12}")

### 1.4 Arithmetic Intensity and `to_dict()`

The `CostProfile` now exposes `arithmetic_intensity` and includes
`total_bytes_accessed` / `bytes_by_precision` in `to_dict()`.

In [ ]:
evo = MPTrotterEvolution(trotter_steps=50, precision_schedule=Precision.FP32)
cost = evo.cost_profile(H)

print(f"Total FLOPs:          {cost.total_flops:,}")
print(f"Total bytes accessed: {cost.total_bytes_accessed:,}")
print(f"Arithmetic intensity: {cost.arithmetic_intensity:.2f} FLOP/B")
print(f"\nBytes by precision:")
for prec, b in cost.bytes_by_precision.items():
    print(f"  {prec}: {b:,} bytes")

# These are also in to_dict() for constraint evaluation
d = cost.to_dict()
print(f"\nNew keys in to_dict(): {[k for k in d if 'byte' in k.lower() or 'intensity' in k.lower()]}")

---

## 2. NVML Hardware Counters

The Legate profiler gives scheduling-level GPU metrics (kernel time, peak VRAM).
The new `NVMLMonitor` complements this with **real-time hardware counters** via
`pynvml`: actual VRAM usage, power draw, GPU/memory utilization, and temperature.

It works as a context manager with a background sampling thread and **silently
no-ops** when pynvml is unavailable (e.g., on a laptop without a GPU).

### 2.1 Basic Usage

In [ ]:
from trovolut.backends.gpu.hardware_counters import NVMLMonitor, nvml_available

print(f"NVML available: {nvml_available()}")

# The monitor works even without a GPU -- it returns zero-filled metrics
with NVMLMonitor(gpu_index=0, sample_interval=0.05) as mon:
    # Simulate some work
    _ = np.linalg.eigvalsh(np.random.randn(500, 500))

metrics = mon.metrics()
print(f"\nSampled {metrics.n_samples} times over {metrics.duration_seconds:.3f}s")
print(f"Peak memory:      {metrics.peak_memory_bytes / 1e6:.1f} MB")
print(f"Avg power:        {metrics.avg_power_watts:.1f} W")
print(f"GPU utilization:  {metrics.avg_gpu_utilization:.1f}%")
print(f"Max temperature:  {metrics.max_temperature_c} C")

### 2.2 Metrics as Dict

All metrics are returned with an `nvml_` prefix, making them safe to merge
into the Legate profiler's metrics dict.

In [ ]:
d = metrics.to_dict()
for k, v in d.items():
    print(f"  {k}: {v}")

### 2.3 NVML Constraints

Three new constraint classes consume the NVML metrics:

- `NVMLPeakMemory(max_bytes)` -- GPU VRAM budget
- `NVMLPowerBudget(max_watts)` -- power envelope
- `NVMLUtilization(target_utilization)` -- higher-is-better (like `LegateConcurrency`)

In [ ]:
from trovolut.backends.gpu.constraints import (
    NVMLPeakMemory,
    NVMLPowerBudget,
    NVMLUtilization,
    Group,
)

# Define constraints for a 300W power-limited A100
mem_c = NVMLPeakMemory(max_bytes=80 * 1024**3, hard=True)  # 80 GB VRAM
pwr_c = NVMLPowerBudget(max_watts=300.0, weight=1.0)
util_c = NVMLUtilization(target_utilization=80.0, weight=0.5)

group = Group([mem_c, pwr_c, util_c])
print(group.generate_llm_instructions())

# Evaluate against synthetic metrics
synthetic = {
    "nvml_peak_memory_bytes": 40 * 1024**3,  # 40 GB
    "nvml_avg_power_watts": 250.0,
    "nvml_avg_gpu_utilization": 72.0,
}
result = group.evaluate(synthetic)
print(f"\nCombined score: {result['combined_score']:.4f}")
for name, info in result['details'].items():
    print(f"  {name}: value={info['value']:.2f}, score={info['score']:.4f}, satisfied={info['satisfied']}")

### 2.4 Enabling NVML in the GPU Backend

When using the real GPU backend, pass `enable_nvml=True` to the `Metrics` class.
The profiler will wrap the Legate subprocess with an `NVMLMonitor` and merge
hardware counter data into the returned metrics dict.

```python
# This requires a real GPU + Legate -- shown for reference
from trovolut.backends.gpu.measure import Metrics

metrics = Metrics(
    constraints=[...],
    test_generator=my_test_gen,
    gpus=4,
    enable_nvml=True,   # <-- new parameter
)
result = metrics(my_dynamics)
# result now contains both Legate and NVML metrics
print(result['nvml_peak_memory_bytes'])  # actual VRAM
print(result['nvml_avg_power_watts'])    # power draw
```

---

## 3. Calibration Infrastructure

Even with the roofline model, emulator predictions won't perfectly match real
hardware. The calibration infrastructure runs paired benchmarks (emulator + optional
real GPU), fits linear correction factors, and stores them for reuse:

$$
\text{actual} \approx \text{slope} \times \text{emulated} + \text{intercept}
$$

This closes the loop for OpenEvolve: optimize cheaply via emulator, with
calibrated predictions of real execution time.

### 3.1 Quick Calibration (Emulator-Only Mode)

Without a GPU, calibration runs in **emulator-only mode** where actual = emulated.
This verifies the pipeline and produces identity correction factors (slope=1, intercept=0).

In [ ]:
from trovolut.calibration import calibrate, CalibrationResult

result = calibrate()  # uses standard_benchmarks() + gpus.A100 by default

print(f"Calibration result ({len(result.emulated_times)} workloads):")
print(f"  Time:   slope={result.time_slope:.4f}, intercept={result.time_intercept:.2e}, R²={result.time_r2:.6f}")
print(f"  Memory: slope={result.memory_slope:.4f}, intercept={result.memory_intercept:.2e}, R²={result.memory_r2:.6f}")

### 3.2 Standard Benchmarks

The `standard_benchmarks()` factory generates 72 workloads varying matrix size,
precision, decomposition method, and step count.

In [ ]:
from trovolut.calibration import standard_benchmarks

workloads = standard_benchmarks()
print(f"Total workloads: {len(workloads)}")
print(f"  Sizes:          4, 8, 16, 32")
print(f"  Precisions:     FP64, FP32, FP16")
print(f"  Decompositions: exact, taylor")
print(f"  Step counts:    1, 10, 50")
print(f"  4 x 3 x 2 x 3 = {4*3*2*3}")

### 3.3 Custom Calibration Suite

You can build a `CalibrationSuite` with your own workloads and an optional
`hardware_metrics` callable for actual GPU measurements.

In [ ]:
from trovolut.calibration import CalibrationSuite
from trovolut.backends.gpu_mp_emulator.dynamics import MPTrotterEvolution

# Build workloads with varying matrix sizes
custom_workloads = []
for n in [8, 16, 32, 64, 128]:
    rng = np.random.default_rng(42 + n)
    A = rng.standard_normal((n, n)) + 1j * rng.standard_normal((n, n))
    H = (A + A.conj().T) / 2
    for steps in [10, 50]:
        dyn = MPTrotterEvolution(trotter_steps=steps)
        custom_workloads.append({"hamiltonian": H, "dynamics": dyn})

suite = CalibrationSuite(
    workloads=custom_workloads,
    hardware=gpus.A100,
    hardware_metrics=None,  # emulator-only
)
result = suite.run()

print(f"Custom calibration ({len(result.emulated_times)} workloads):")
print(f"  Time range:   [{min(result.emulated_times):.2e}, {max(result.emulated_times):.2e}] s")
print(f"  Memory range: [{min(result.emulated_memory):.0f}, {max(result.emulated_memory):.0f}] bytes")

### 3.4 Corrected Cost Profiles

Wrap any `CostProfile` with a `CorrectedCostProfile` to apply calibration
corrections. This is transparent to downstream code -- all other attributes
are delegated to the underlying profile.

In [ ]:
from trovolut.calibration import CorrectedCostProfile, CalibrationResult

# Simulate a calibration with a known correction
cal = CalibrationResult(
    time_slope=1.3,       # actual is 30% slower than emulated
    time_intercept=0.001, # plus 1ms fixed overhead
    memory_slope=1.1,     # 10% more memory
    memory_intercept=1024,
)

# Get a cost profile from any evolution
evo = MPTrotterEvolution(trotter_steps=50)
n = 32
rng = np.random.default_rng(42)
A = rng.standard_normal((n, n)) + 1j * rng.standard_normal((n, n))
H = (A + A.conj().T) / 2
profile = evo.cost_profile(H)

# Wrap with calibration
corrected = CorrectedCostProfile(profile, cal)

raw_time = profile.emulated_time(gpus.A100)
cor_time = corrected.corrected_time(gpus.A100)
raw_mem = profile.peak_memory_bytes
cor_mem = corrected.corrected_memory()

print(f"{'':20s} {'Emulated':>14s} {'Corrected':>14s}")
print("-" * 50)
print(f"{'Time (s)':<20s} {raw_time:>14.2e} {cor_time:>14.2e}")
print(f"{'Memory (bytes)':<20s} {raw_mem:>14,} {cor_mem:>14,.0f}")
print(f"\n(Transparent delegation: corrected.method = '{corrected.method}')")

### 3.5 JSON Persistence

Calibration results can be saved and loaded for reuse across sessions.

In [ ]:
import tempfile, os

# Save to a temporary file
with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
    f.write(cal.to_json())
    path = f.name

# Load it back
with open(path) as f:
    loaded = CalibrationResult.from_json(f.read())

print(f"Roundtrip check:")
print(f"  time_slope:     {cal.time_slope} -> {loaded.time_slope}")
print(f"  time_intercept: {cal.time_intercept} -> {loaded.time_intercept}")
print(f"  memory_slope:   {cal.memory_slope} -> {loaded.memory_slope}")

os.unlink(path)

### 3.6 One-Line Calibration with `calibrate()`

The top-level `calibrate()` convenience function is also exported from
`trovolut` directly.

In [ ]:
# Available as trv.calibrate()
result = trv.calibrate()
print(f"trv.calibrate() ran {len(result.emulated_times)} benchmarks")
print(f"  R² (time):   {result.time_r2:.6f}")
print(f"  R² (memory): {result.memory_r2:.6f}")

---

## 4. Putting It All Together

A typical workflow combining all three features:

1. **Calibrate** the emulator against your hardware (once, save JSON)
2. Use the **roofline model** to get bandwidth-aware cost estimates
3. Wrap cost profiles with **CorrectedCostProfile** for calibrated predictions
4. Monitor real runs with **NVMLMonitor** for validation

```python
import trovolut as trv
trv.set_backend('gpu-mp-emulator')

from trovolut.calibration import calibrate, CorrectedCostProfile
from trovolut.backends.gpu.hardware_counters import NVMLMonitor

# Step 1: Calibrate (do this once)
cal = calibrate(save_path='calibration.json')

# Step 2: Get roofline-aware cost profile
evo = trv.dynamics.MPTrotterEvolution(trotter_steps=100)
profile = evo.cost_profile(hamiltonian)
print(f'Arithmetic intensity: {profile.arithmetic_intensity:.1f} FLOP/B')

# Step 3: Apply calibration correction
corrected = CorrectedCostProfile(profile, cal)
print(f'Corrected time: {corrected.corrected_time(hardware):.3f}s')

# Step 4: Validate with NVML (on real GPU)
with NVMLMonitor() as mon:
    psi = evo.evolve(hamiltonian, psi0, t)
m = mon.metrics()
print(f'Actual VRAM: {m.peak_memory_bytes / 1e9:.1f} GB')
print(f'Power draw:  {m.avg_power_watts:.0f} W')
```

## Summary

| Feature | Key Classes | What It Adds |
|---------|-------------|-------------|
| Roofline model | `CostProfile`, `SpacetimeCost` | `total_bytes_accessed`, `arithmetic_intensity`, bandwidth-aware `emulated_time()` |
| NVML counters | `NVMLMonitor`, `HardwareCounterMetrics` | Real-time VRAM, power, utilization, temperature |
| NVML constraints | `NVMLPeakMemory`, `NVMLPowerBudget`, `NVMLUtilization` | Power/memory budgets for optimization |
| Calibration | `CalibrationSuite`, `CorrectedCostProfile`, `calibrate()` | Linear correction factors, JSON persistence |

For the full mixed-precision optimization workflow, see the
[GPU Optimization](mp_optimization.ipynb) tutorial.